# Explanation Generation
I need to use the model to get *k* explanations for each line of generated code. Firtly, I need to delete the comments line from my dataset.

In [1]:
import re
import os
import pandas as pd
from tqdm import tqdm
from enum import Enum
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
python_csv_file = '/datasets/leetcode_selected_problems_python_14B_no_comments.csv'
c_sharp_csv_file = '/datasets/leetcode_selected_problems_c_sharp_14B_no_comments.csv'
typescript_csv_file = '/datasets/leetcode_selected_problems_typescript_14B_no_comments.csv'

def get_generated_code(csv_file):
    csv_full_path = os.getcwd()+csv_file
    if os.path.exists(csv_full_path):
        return pd.read_csv(csv_full_path)
    else:
        print("File not found.")
        return None
    
python_solutions = get_generated_code(python_csv_file)
c_sharp_solutions = get_generated_code(c_sharp_csv_file)
typescript_solutions = get_generated_code(typescript_csv_file)

In [3]:
class Language(Enum):
    PYTHON = "python"
    TYPESCRIPT = "typescript"
    C_SHARP = "csharp"

In [4]:
model_name = "Qwen/Qwen2.5-Coder-14B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, temperature=0)

2025-03-18 21:10:09.730650: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-18 21:10:09.768534: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-18 21:10:09.768550: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-18 21:10:09.768573: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-18 21:10:09.776024: I tensorflow/core/platform/cpu_feature_g

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [23]:
def get_generated_explanations(programming_language, problem_description, code_solution):
    explanation_prompt = f"""
        Read the given problem and the proposed {programming_language} solution. Please output strictly following the explanation-only format for EACH line:

        [Line x]: Explanation: <your explanation>
        
        Problem: {problem_description}
        Code: ```{code_solution}```
        """
    explanation_inputs = tokenizer(explanation_prompt, return_tensors="pt", padding=True)
    explanation_outputs = model.generate(input_ids=explanation_inputs['input_ids'], attention_mask=explanation_inputs['attention_mask'], max_length=2048)
    return tokenizer.decode(explanation_outputs[0], skip_special_tokens=True)[len(explanation_prompt)-1:]

In [19]:
columns = ['title', 'explanation']
python_explanations = pd.DataFrame([], columns=columns)
c_sharp_explanations = pd.DataFrame([], columns=columns)
typescript_explanations = pd.DataFrame([], columns=columns)

In [20]:
def add_explanations(solutions, programming_language, explanations):
    for index, row in tqdm(solutions.iterrows()):
         explanations.loc[len(explanations)] = [row['title'], get_generated_explanations(programming_language, row['description'], row['solution_no_comments'])]

In [21]:
python_solutions[0:1]

,title,description,difficulty,solution,solution_no_comments
0,Relative Ranks,You are given an integer array `score` of size...,Easy,def findRelativeRanks(score):\n # Step 1: C...,def findRelativeRanks(score):\n sorted_scor...


In [22]:
print(python_solutions[0:1]["solution_no_comments"].to_list()[0])

def findRelativeRanks(score):
    sorted_scores = score[:]
    n = len(sorted_scores)
    for i in range(n):
        for j in range(0, n-i-1):
            if sorted_scores[j] < sorted_scores[j+1]:
                sorted_scores[j], sorted_scores[j+1] = sorted_scores[j+1], sorted_scores[j]
    rank_dict = {}
    for idx, value in enumerate(sorted_scores):
        if idx == 0:
            rank_dict[value] = "Gold Medal"
        elif idx == 1:
            rank_dict[value] = "Silver Medal"
        elif idx == 2:
            rank_dict[value] = "Bronze Medal"
        else:
            rank_dict[value] = str(idx + 1)
    result = []
    for s in score:
        result.append(rank_dict[s])
    return result
print(findRelativeRanks([5, 4, 3, 2, 1]))  
print(findRelativeRanks([10, 3, 8, 9, 4]))


In [24]:
add_explanations(python_solutions[0:1], Language.PYTHON, python_explanations)

1it [14:01, 841.99s/it]


In [25]:
python_explanations

,title,explanation
0,Relative Ranks,[Line 1]: Explanation: The function `findRel...


In [26]:
print(python_explanations[0:1]["explanation"].to_list()[0])

  [Line 1]: Explanation: The function `findRelativeRanks` takes an array `score` as input, which represents the scores of athletes in a competition.
        [Line 2]: Explanation: A copy of the `score` array is made and stored in `sorted_scores`. This will be used to sort the scores in descending order.
        [Line 3]: Explanation: The length of the `sorted_scores` array is stored in the variable `n`.
        [Line 4]: Explanation: A nested loop is initiated to perform a bubble sort on the `sorted_scores` array. The outer loop runs `n` times, and the inner loop compares adjacent elements and swaps them if they are in the wrong order.
        [Line 5]: Explanation: The inner loop runs from the start of the array to `n-i-1` to ensure that the largest unsorted element bubbles up to its correct position in each iteration.
        [Line 6]: Explanation: If the current element is smaller than the next element, they are swapped.
        [Line 7]: Explanation: After sorting, a dictionary `ra

In [17]:
python_explanations.to_csv('datasets/leetcode_selected_problems_python_explanations.csv', index=False)